---

# 🎓 Self-Assignment: Build Your Own RAG System

## Mini Project — "Ask My Documents"

**Objective:** Build a complete, end-to-end RAG application using everything you learned in this course. You will create a system that can answer questions about **your own documents** (PDFs, text files, or web pages).

**Estimated Time:** 2–3 hours

**Difficulty:** ⭐⭐⭐ Intermediate

---

### 📋 Project Brief

> *You are an AI engineer at a company. Your team has a collection of internal documents (policies, technical guides, meeting notes). Build a RAG-powered Q&A system that lets employees ask natural language questions and get accurate, grounded answers from these documents.*

---

### ✅ Requirements

Your submission must include a working Jupyter notebook with the following **7 tasks**:

| Task | Description | Points |
|------|-------------|--------|
| **Task 1** | Load at least **3 documents** from at least **2 different sources** (e.g., PDF + Web, or PDF + TXT) | 10 |
| **Task 2** | Implement a chunking strategy with a **justified choice** of `chunk_size` and `chunk_overlap` — write a comment explaining your reasoning | 10 |
| **Task 3** | Store embeddings in a **Chroma** vector store with persistence enabled | 10 |
| **Task 4** | Build a **retriever** and demonstrate it works by showing top-K results for 3 different queries | 10 |
| **Task 5** | Write a **custom RAG prompt** that includes a system message with specific instructions (e.g., "answer in bullet points", "cite the source page", or "say I don't know if unsure") | 15 |
| **Task 6** | Assemble the **complete RAG chain** using LCEL (pipe operators) and test it with at least **5 questions** | 15 |
| **Task 7** | **Bonus Challenges** (pick at least one) — see below | 30 |

**Total: 100 points**

---

### 🌟 Task 7 — Bonus Challenges (pick at least one for full marks)

| Challenge | Description | Points |
|-----------|-------------|--------|
| **A. Multi-turn memory** | Add conversation history so follow-up questions work (e.g., "What about its pricing?" after asking about a product) | 10 |
| **B. Source citation** | Modify the chain to return **which documents** were used to answer each question (page number, filename) | 10 |
| **C. Evaluation harness** | Create 5 question-answer pairs as ground truth, then run your RAG chain and compare its answers against the ground truth (simple string match or LLM-as-judge) | 10 |
| **D. Chunking experiment** | Try 3 different chunk sizes (e.g., 200, 500, 1000) and compare retrieval quality — which size finds the best context for the same query? | 10 |
| **E. Hybrid retriever** | Combine similarity search with MMR or metadata filtering — explain when each strategy is better | 10 |
| **F. Streaming UI** | Build a simple interactive cell where users type questions and see the RAG chain stream its answer token by token | 10 |

---

### 🏗️ Starter Scaffold

The cells below provide a **skeleton structure** for your project. Each cell has `TODO` comments where you need to fill in your code. The structure follows the same 4-phase approach from the course.

> **Tip:** Refer back to the demo cells above (Cells 1–22) whenever you get stuck. The patterns are the same — you're just applying them to your own data now.

---

#### Task 1 — Load Your Documents

In [ ]:
import os
from google.colab import userdata
from langchain_community.document_loaders import WebBaseLoader

# Configure Google API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# ── Source: Wikipedia for Trails of Cold Steel 1, 2, 3, and 4 ──
web_loader = WebBaseLoader([
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel",
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel_II",
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel_III",
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel_IV"
])
all_docs = web_loader.load()

print(f"🌐 Web documents loaded: {len(all_docs)}")
print(f"📦 Total: {len(all_docs)}")

#### Task 2 — Chunk Your Documents

Choose your `chunk_size` and `chunk_overlap` and **explain why** in a comment.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# I chose chunk_size=800 and overlap=150.
# Reasoning: JRPG lore is dense with character names and plot points.
# Larger chunks (800) help maintain the context of specific story arcs,
# while the 150 overlap ensures continuity across detailed story descriptions.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)

chunks = splitter.split_documents(all_docs)

print(f"📄 Original documents: {len(all_docs)}")
print(f"✂️  Chunks created:     {len(chunks)}")

#### Task 3 — Store in ChromaDB

In [ ]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import shutil, os

# Initialize Google Gemini Embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

PERSIST_DIR = "./cold_steel_db"
if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="cold_steel_lore",
)

print(f"✅ Stored {len(chunks)} chunks in Chroma using Gemini Embeddings")

#### Task 4 — Build & Test the Retriever

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

test_queries = [
    "Who is Rean Schwarzer?",
    "What is the setting of the Erebonian Empire?",
    "How does the combat system work?",
]

for query in test_queries:
    docs = retriever.invoke(query)
    print(f"\n🔎 Query: \"{query}\"")
    for i, doc in enumerate(docs):
        print(f"   [{i+1}] {doc.page_content[:100]}...")

#### Task 5 — Design Your Custom RAG Prompt

Create a **custom system message** that shapes how the model answers. Be creative and specific.

In [ ]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

# Custom prompt with Persona and Citation instructions
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert on 'The Legend of Heroes: Trails of Cold Steel' series.
    Answer the question using the provided context.
    - Cite your sources clearly using the URL from the metadata.
    - If you don't know the answer, state that the information is not in the records.
    - Use bullet points for characters or gameplay mechanics."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

#### Task 6 — Assemble & Test the Full RAG Chain

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# Initialize Gemini Pro
model = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)

# Helper to include Source Citations (Challenge B)
def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata.get('source')}] {d.page_content}" for d in docs)

rag_chain = (
    RunnableParallel(
        context = retriever | format_docs,
        question = RunnablePassthrough(),
        chat_history = lambda x: x.get("chat_history", [])
    )
    | rag_prompt
    | model
    | StrOutputParser()
)

# Test with 5 Questions
questions = [
    "What is Class VII?",
    "Who are the main protagonists?",
    "Explain the setting of the game.",
    "What are Tactical Link systems?",
    "Which empire does the story take place in?"
]

for q in questions:
    print(f"\n❓ {q}\n💡 {rag_chain.invoke({'question': q, 'chat_history': []})}\n" + "-"*30)

#### Task 7 — Bonus Challenge(s)

Pick **at least one** bonus challenge from the table above and implement it below.

In [ ]:
import time
from langchain_core.messages import HumanMessage, AIMessage

print("--- Challenge A: Multi-turn Memory ---")
chat_history = []
q1 = "Who is the protagonist?"
a1 = rag_chain.invoke({"question": q1, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=q1), AIMessage(content=a1)])
print(f"Q: {q1}\nA: {a1[:100]}...")

q2 = "What is his special weapon?"
a2 = rag_chain.invoke({"question": q2, "chat_history": chat_history})
print(f"\nFollow-up Q: {q2}\nA: {a2[:100]}...")

print("\n--- Challenge F: Streaming UI ---")
query = "Summarize the plot of the first game."
print(f"Streaming response for: {query}\n")
for chunk in rag_chain.stream({"question": query, "chat_history": []}):
    print(chunk, end="", flush=True)
    time.sleep(0.01)

---

### 📦 Submission Checklist

Before submitting, verify the following:

- [ ] **Task 1:** Loaded 3+ documents from 2+ different source types
- [ ] **Task 2:** Chunking strategy implemented with a written justification comment
- [ ] **Task 3:** Chroma vector store created with persistence to disk
- [ ] **Task 4:** Retriever tested with 3 queries, showing top-K results with metadata
- [ ] **Task 5:** Custom RAG prompt with a specific, thoughtful system message
- [ ] **Task 6:** Full RAG chain assembled with LCEL pipes, tested with 5+ questions
- [ ] **Task 7:** At least one bonus challenge completed
- [ ] **All cells run** top-to-bottom without errors (Kernel → Restart & Run All)
- [ ] **No hardcoded API keys** — uses environment variables or `.env` file

### 📁 What to Submit

1. This notebook (`.ipynb`) with all cells executed and outputs visible
2. Your `.env.example` file (with placeholder values, NOT real keys)
3. A short `README.md` (3–5 sentences) describing:
   - What documents you chose and why
   - What bonus challenge(s) you completed
   - One thing you learned or found surprising

---

### 💡 Tips for Success

- **Start with small, simple documents** — don't load 500 pages on your first try.
- **Test each task independently** before chaining them together.
- **Print intermediate results** — check what your retriever returns before building the full chain.
- **Experiment with chunk sizes** — this is the single biggest lever for RAG quality.
- **Read the error messages** — LangChain errors are usually descriptive and tell you exactly what's wrong.

Good luck! 🚀

---

## 🧹 Cleanup

In [ ]:
# ============================================================
# CLEANUP: Remove persisted Chroma database
# ============================================================
import shutil, os

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("🗑️  Removed './chroma_db/' directory")
else:
    print("ℹ️  Nothing to clean up")

print("\n✅ Demo complete! You've built a full RAG pipeline with Azure OpenAI + ChromaDB + LangChain.")